In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

In [2]:
X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
y_train = pd.read_csv("data/y_train.csv").values.ravel()
y_test = pd.read_csv("data/y_test.csv").values.ravel()

In [3]:
rare_classes = ["type", "hover", "scrolldown", "appear", "disappear", "scrollup"]
y_train = pd.Series(y_train).replace(rare_classes, "other").values
y_test = pd.Series(y_test).replace(rare_classes, "other").values

In [4]:
classes = np.sort(np.unique(y_train))
print("Classes:", classes)
print(f"Number of classes: {len(classes)}")

Classes: ['click' 'drag' 'other' 'select' 'zoomin' 'zoomout']
Number of classes: 6


In [8]:
pipeline = Pipeline([
    ("smote", SMOTE()),
    ("rf", RandomForestClassifier())
])

param_grid = {
    "rf__n_estimators": [100, 200, 300, 500],
    "rf__max_depth": [None, 10, 20, 30, 50],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4],
    "rf__max_features": ["sqrt", "log2"],
    "rf__class_weight": [None, "balanced"],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

Fitting 5 folds for each of 720 candidates, totalling 3600 fits
[CV] END rf__class_weight=None, rf__max_depth=None, rf__max_features=sqrt, rf__min_samples_leaf=1, rf__min_samples_split=2, rf__n_estimators=100; total time=  15.9s
[CV] END rf__class_weight=None, rf__max_depth=None, rf__max_features=sqrt, rf__min_samples_leaf=1, rf__min_samples_split=2, rf__n_estimators=100; total time=  17.2s
[CV] END rf__class_weight=None, rf__max_depth=None, rf__max_features=sqrt, rf__min_samples_leaf=1, rf__min_samples_split=2, rf__n_estimators=100; total time=  17.5s
[CV] END rf__class_weight=None, rf__max_depth=None, rf__max_features=sqrt, rf__min_samples_leaf=1, rf__min_samples_split=2, rf__n_estimators=100; total time=  17.9s
[CV] END rf__class_weight=None, rf__max_depth=None, rf__max_features=sqrt, rf__min_samples_leaf=1, rf__min_samples_split=2, rf__n_estimators=100; total time=  19.2s
[CV] END rf__class_weight=None, rf__max_depth=None, rf__max_features=sqrt, rf__min_samples_leaf=1, rf__min_samp

In [9]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("Multi-class Random Forest Results:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f} (weighted)")
print(f"  Recall:    {recall:.4f} (weighted)")
print(f"  F1:        {f1:.4f} (weighted)")

print("Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

Multi-class Random Forest Results:
  Accuracy:  0.6856
  Precision: 0.7315 (weighted)
  Recall:    0.6856 (weighted)
  F1:        0.6936 (weighted)
Classification report:
              precision    recall  f1-score   support

       click       0.59      0.72      0.65       291
        drag       0.84      0.60      0.70       426
       other       0.47      0.80      0.59        20
      select       0.25      0.61      0.36        23
      zoomin       0.78      0.91      0.84        70
     zoomout       0.86      0.84      0.85        67

    accuracy                           0.69       897
   macro avg       0.63      0.75      0.66       897
weighted avg       0.73      0.69      0.69       897

